# Bare Metal Rebuild — Notebook Version

This is the notebook-native version of the bare-metal rebuild pipeline.

It keeps everything inline and executable in notebook flow:

- SHA-256 core operators
- schedule forward expansion
- 16-step carry-free backsolve from $W_{16..31}$ to $W_{0..15}$
- seam / carry / seed-$B$ extraction
- `Sigma0(a) == Maj(a,b,c)` jumper diagnostic
- GF(2) Jacobian / nullspace / parity-check machinery
- seam augmentation tests

The working hardware grammar is:

$$
\text{RAIL} \to \text{INJECT} \to \text{FOLD} \to \text{OVERLAP} \to \text{CARRY} \to \text{RETAIN} \to \text{PROJECT} \to \text{VERIFY}
$$

and the key seed is

$$
B = \Sigma_0(a)\land \operatorname{Maj}(a,b,c).
$$


## 1. Core setup

This cell defines the SHA-256 state rails, rotations, boolean primitives, and all notebook-native helpers.


In [1]:

import json
import math
from typing import List, Tuple
import numpy as np

MASK32 = 0xFFFFFFFF

H0 = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]

K = [
    0x428A2F98, 0x71374491, 0xB5C0FBCF, 0xE9B5DBA5, 0x3956C25B, 0x59F111F1, 0x923F82A4, 0xAB1C5ED5,
    0xD807AA98, 0x12835B01, 0x243185BE, 0x550C7DC3, 0x72BE5D74, 0x80DEB1FE, 0x9BDC06A7, 0xC19BF174,
    0xE49B69C1, 0xEFBE4786, 0x0FC19DC6, 0x240CA1CC, 0x2DE92C6F, 0x4A7484AA, 0x5CB0A9DC, 0x76F988DA,
    0x983E5152, 0xA831C66D, 0xB00327C8, 0xBF597FC7, 0xC6E00BF3, 0xD5A79147, 0x06CA6351, 0x14292967,
    0x27B70A85, 0x2E1B2138, 0x4D2C6DFC, 0x53380D13, 0x650A7354, 0x766A0ABB, 0x81C2C92E, 0x92722C85,
    0xA2BFE8A1, 0xA81A664B, 0xC24B8B70, 0xC76C51A3, 0xD192E819, 0xD6990624, 0xF40E3585, 0x106AA070,
    0x19A4C116, 0x1E376C08, 0x2748774C, 0x34B0BCB5, 0x391C0CB3, 0x4ED8AA4A, 0x5B9CCA4F, 0x682E6FF3,
    0x748F82EE, 0x78A5636F, 0x84C87814, 0x8CC70208, 0x90BEFFFA, 0xA4506CEB, 0xBEF9A3F7, 0xC67178F2,
]

def u32(x: int) -> int:
    return x & MASK32

def rotr(x: int, n: int) -> int:
    x &= MASK32
    return ((x >> n) | ((x << (32 - n)) & MASK32)) & MASK32

def Sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def Sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ (x >> 3)

def sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ (x >> 10)

def Ch(e: int, f: int, g: int) -> int:
    return u32((e & f) ^ ((~e) & g))

def Maj(a: int, b: int, c: int) -> int:
    return u32((a & b) ^ (a & c) ^ (b & c))

def parse_hex_words(text: str, n: int) -> List[int]:
    parts = [p.strip() for p in text.replace("\n", ",").split(",") if p.strip()]
    if len(parts) != n:
        raise ValueError(f"expected {n} words, got {len(parts)}")
    out = []
    for p in parts:
        q = p.lower()
        if q.startswith("0x"):
            q = q[2:]
        out.append(int(q, 16) & MASK32)
    return out

def words_to_hex(words: List[int]) -> List[str]:
    return [f"0x{w:08x}" for w in words]


## 2. Schedule engine

This is the forward expansion and the reverse 16-step rebuild:

- `expand_schedule_from_w0_15(...)`
- `rebuild_w0_15_from_w16_31(...)`


In [2]:

def expand_schedule_from_w0_15(w0_15: List[int]) -> List[int]:
    if len(w0_15) != 16:
        raise ValueError("need 16 words for W0..15")
    W = [u32(w) for w in w0_15] + [0] * 48
    for t in range(16, 64):
        W[t] = u32(sigma1(W[t - 2]) + W[t - 7] + sigma0(W[t - 15]) + W[t - 16])
    return W

def rebuild_w0_15_from_w16_31(w16_31: List[int]) -> Tuple[List[int], List[int]]:
    if len(w16_31) != 16:
        raise ValueError("need 16 words for W16..31")
    W = [0] * 32
    for i, w in enumerate(w16_31, start=16):
        W[i] = u32(w)
    for t in range(31, 15, -1):
        W[t - 16] = u32(W[t] - sigma1(W[t - 2]) - W[t - 7] - sigma0(W[t - 15]))
    return W[:16], W


## 3. Round engine and hardware observables

This cell defines:

- round terms
- state transition
- seam XOR
- seam full
- carry residual
- universal seed $B$
- jumper diagnostic $\Sigma_0(a) == \operatorname{Maj}(a,b,c)$


In [3]:

def round_terms(state: List[int], w: int, r: int) -> Tuple[int, int]:
    a, b, c, d, e, f, g, h = state
    t1 = u32(h + Sigma1(e) + Ch(e, f, g) + K[r] + w)
    t2 = u32(Sigma0(a) + Maj(a, b, c))
    return t1, t2

def round_step(state: List[int], w: int, r: int) -> List[int]:
    a, b, c, d, e, f, g, h = state
    t1, t2 = round_terms(state, w, r)
    return [
        u32(t1 + t2),
        a,
        b,
        c,
        u32(d + t1),
        e,
        f,
        g,
    ]

def seam_xor_word(state: List[int]) -> int:
    a, b, c, d, e, f, g, h = state
    return u32(Sigma0(a) ^ Maj(a, b, c) ^ d)

def seam_full_word(state: List[int]) -> int:
    a, b, c, d, e, f, g, h = state
    return u32(Sigma0(a) + Maj(a, b, c) + d)

def carry_residual_word(state: List[int]) -> int:
    return u32(seam_full_word(state) ^ seam_xor_word(state))

def seed_B_word(state: List[int]) -> int:
    a, b, c, d, e, f, g, h = state
    return u32(Sigma0(a) & Maj(a, b, c))

def jumper_hit(state: List[int]) -> bool:
    a, b, c, d, e, f, g, h = state
    return Sigma0(a) == Maj(a, b, c)

def observe_six_rounds(w6: List[int], state0: List[int] = None):
    state = (state0 or H0).copy()
    rows = []
    for r in range(6):
        rows.append({
            "r": r,
            "state": state.copy(),
            "seam_xor": seam_xor_word(state),
            "seam_full": seam_full_word(state),
            "carry_residual": carry_residual_word(state),
            "seed_B": seed_B_word(state),
            "jump": jumper_hit(state),
        })
        state = round_step(state, w6[r], r)
    return rows


## 4. Quick sanity checks

This proves the basic anchors are alive in the notebook.


In [4]:

# Ground witness
t1_0, t2_0 = round_terms(H0.copy(), 0, 0)
print(f"T2_0^(0) = 0x{t2_0:08x}")
assert t2_0 == 0x08909AE5

# First-step displacement
for W0 in [0, 1, 0x80000000, 0x12345678, 0xFFFFFFFF]:
    base = round_step(H0.copy(), 0, 0)
    live = round_step(H0.copy(), W0, 0)
    t1_base, _ = round_terms(H0.copy(), 0, 0)
    t1_live, _ = round_terms(H0.copy(), W0, 0)
    assert ((t1_live - t1_base) & MASK32) == (W0 & MASK32)
    assert ((live[0] - base[0]) & MASK32) == (W0 & MASK32)
    assert ((live[4] - base[4]) & MASK32) == (W0 & MASK32)

print("Verified:")
print("  • T2_0^(0) ground witness")
print("  • T1_0 - T1_0^(0) = W0")
print("  • first-step displacement into a/e")


T2_0^(0) = 0x08909ae5
Verified:
  • T2_0^(0) ground witness
  • T1_0 - T1_0^(0) = W0
  • first-step displacement into a/e


## 5. Orbit workbench

Put six words into `w6_text` and run the next cell.


In [5]:

w6_text = "0x00000000,0x00000000,0x00000000,0x00000000,0x00000000,0x00000000"
w6 = parse_hex_words(w6_text, 6)
rows = observe_six_rounds(w6)

print("r  seam_xor    seam_full   carry_res   seed_B      jump")
for row in rows:
    print(
        f"{row['r']:<2d} "
        f"{row['seam_xor']:08x}  "
        f"{row['seam_full']:08x}  "
        f"{row['carry_residual']:08x}  "
        f"{row['seed_B']:08x}  "
        f"{row['jump']}"
    )


r  seam_xor    seam_full   carry_res   seed_B      jump
0  5100a723  ade0901f  fce0373c  0a20a466  False
1  d92ba890  55c5975e  8cee3fce  1a08a405  False
2  aed18344  a531604e  0be0e30a  6a09c204  False
3  7868b451  4d9b88af  35f33cfe  e8982809  False
4  1bca3d90  1405be6e  0fcf83fe  181d4022  False
5  ce98cfed  c4539011  0acb5ffc  4a9c4002  False


## 6. Forward schedule example

Put 16 words into `w0_15_text` and run the next cell.


In [6]:

w0_15_text = ",".join(["0x00000000"] * 16)
w0_15 = parse_hex_words(w0_15_text, 16)
W = expand_schedule_from_w0_15(w0_15)

print("W0_15:")
print(words_to_hex(W[:16]))
print("\nW16_31:")
print(words_to_hex(W[16:32]))
print("\nW32_63:")
print(words_to_hex(W[32:64]))


W0_15:
['0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000']

W16_31:
['0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000']

W32_63:
['0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000']


## 7. Reverse rebuild example

This takes `W16..31` and rebuilds `W0..15` with the 16 carry-free subtractions.


In [7]:

w16_31 = W[16:32]
rebuilt_w0_15, rebuilt_w0_31 = rebuild_w0_15_from_w16_31(w16_31)

print("Original W0_15:")
print(words_to_hex(W[:16]))
print("\nRebuilt W0_15:")
print(words_to_hex(rebuilt_w0_15))
print("\nMatch:", [u32(a) == u32(b) for a, b in zip(W[:16], rebuilt_w0_15)])
assert all(u32(a) == u32(b) for a, b in zip(W[:16], rebuilt_w0_15))


Original W0_15:
['0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000']

Rebuilt W0_15:
['0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000', '0x00000000']

Match: [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]


## 8. GF(2) machinery

This is the full notebook-native linear algebra layer:

- rank
- nullspace
- left nullspace
- one particular solve


In [8]:

def bits_of_word(x: int, width: int = 32) -> np.ndarray:
    return np.array([(x >> i) & 1 for i in range(width)], dtype=np.uint8)

def flatten_words(words: List[int]) -> np.ndarray:
    return np.concatenate([bits_of_word(w) for w in words], axis=0)

def seam_observable_from_w6(w6: List[int], basis: str = "seam_xor") -> np.ndarray:
    rows = observe_six_rounds(w6)
    if basis == "seam_xor":
        words = [row["seam_xor"] for row in rows]
    elif basis == "seam_full":
        words = [row["seam_full"] for row in rows]
    elif basis == "carry_residual":
        words = [row["carry_residual"] for row in rows]
    elif basis == "seed_B":
        words = [row["seed_B"] for row in rows]
    elif basis == "seam_plus_d":
        words = []
        for row in rows:
            words.append(row["seam_xor"])
            words.append(row["state"][3])
    elif basis == "seam_plus_a":
        words = []
        for row in rows:
            words.append(row["seam_xor"])
            words.append(row["state"][0])
    elif basis == "seam_plus_e":
        words = []
        for row in rows:
            words.append(row["seam_xor"])
            words.append(row["state"][4])
    else:
        raise ValueError(f"unknown basis {basis}")
    return flatten_words(words)

def build_jacobian_w6(basis: str = "seam_xor") -> np.ndarray:
    y0 = seam_observable_from_w6([0] * 6, basis=basis)
    J = np.zeros((len(y0), 192), dtype=np.uint8)
    col = 0
    for j in range(6):
        for b in range(32):
            w6 = [0] * 6
            w6[j] = 1 << b
            y = seam_observable_from_w6(w6, basis=basis)
            J[:, col] = y0 ^ y
            col += 1
    return J

def gf2_rref(A: np.ndarray):
    A = (A.copy() & 1).astype(np.uint8)
    m, n = A.shape
    pivots = []
    row = 0
    for col in range(n):
        pivot = None
        for r in range(row, m):
            if A[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != row:
            A[[row, pivot]] = A[[pivot, row]]
        for r in range(m):
            if r != row and A[r, col]:
                A[r] ^= A[row]
        pivots.append(col)
        row += 1
        if row == m:
            break
    return A, pivots

def gf2_rank(A: np.ndarray) -> int:
    _, pivots = gf2_rref(A)
    return len(pivots)

def gf2_nullspace_basis(A: np.ndarray) -> np.ndarray:
    R, pivots = gf2_rref(A)
    m, n = R.shape
    pivot_set = set(pivots)
    free_cols = [j for j in range(n) if j not in pivot_set]
    basis = []
    for fc in free_cols:
        v = np.zeros(n, dtype=np.uint8)
        v[fc] = 1
        for i, pc in enumerate(pivots):
            if R[i, fc]:
                v[pc] = 1
        basis.append(v)
    if not basis:
        return np.zeros((n, 0), dtype=np.uint8)
    return np.stack(basis, axis=1)

def gf2_left_nullspace_basis(A: np.ndarray) -> np.ndarray:
    return gf2_nullspace_basis(A.T)

def gf2_solve_particular(A: np.ndarray, b: np.ndarray):
    A = (A.copy() & 1).astype(np.uint8)
    b = (b.copy() & 1).astype(np.uint8).reshape(-1, 1)
    M = np.concatenate([A, b], axis=1)
    m, n1 = M.shape
    n = n1 - 1
    row = 0
    pivots = []
    for col in range(n):
        pivot = None
        for r in range(row, m):
            if M[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != row:
            M[[row, pivot]] = M[[pivot, row]]
        for r in range(m):
            if r != row and M[r, col]:
                M[r] ^= M[row]
        pivots.append(col)
        row += 1
        if row == m:
            break
    for r in range(m):
        if not M[r, :n].any() and M[r, n]:
            return None
    x = np.zeros(n, dtype=np.uint8)
    for i, col in enumerate(pivots):
        x[col] = M[i, n]
    return x


## 9. Geometry basis selector

Pick the observable basis and run the next cell.

Choices:
- `"seam_xor"`
- `"seam_full"`
- `"carry_residual"`
- `"seed_B"`
- `"seam_plus_d"`
- `"seam_plus_a"`
- `"seam_plus_e"`


In [9]:

basis = "seam_xor"

J = build_jacobian_w6(basis)
rank = gf2_rank(J)
N = gf2_nullspace_basis(J)
H = gf2_left_nullspace_basis(J).T

print("basis =", basis)
print("shape =", J.shape)
print("rank =", rank)
print("nullity =", J.shape[1] - rank)
print("left-nullity =", J.shape[0] - rank)
print("nullspace basis shape =", N.shape)
print("parity check shape =", H.shape)


basis = seam_xor
shape = (192, 192)
rank = 157
nullity = 35
left-nullity = 35
nullspace basis shape = (192, 35)
parity check shape = (35, 192)


## 10. Linearized parity-recovery scaffold

This is the notebook version of the parity-check / branch stage.


In [10]:

target_x = np.zeros(192, dtype=np.uint8)
target_x[0] = 1

target_y = (J @ target_x) & 1
parity_ok = np.all(((H @ target_y) & 1) == 0) if H.size else True
x_particular = gf2_solve_particular(J, target_y)

print("target passes parity?", parity_ok)
print("particular solution found?", x_particular is not None)
if x_particular is not None:
    print("particular solution weight =", int(x_particular.sum()))


target passes parity? True
particular solution found? True
particular solution weight = 1


## 11. Augmented seam rank test

This is the same closure test you were driving toward:

- seam only
- seam + d
- seam + a
- seam + e


In [11]:

for basis in ["seam_xor", "seam_plus_d", "seam_plus_a", "seam_plus_e"]:
    Jb = build_jacobian_w6(basis)
    rb = gf2_rank(Jb)
    print(f"{basis:12s} -> shape {Jb.shape}, rank {rb}/{Jb.shape[1]}, nullity {Jb.shape[1]-rb}")


seam_xor     -> shape (192, 192), rank 157/192, nullity 35
seam_plus_d  -> shape (384, 192), rank 158/192, nullity 34
seam_plus_a  -> shape (384, 192), rank 160/192, nullity 32
seam_plus_e  -> shape (384, 192), rank 160/192, nullity 32


## 12. Notebook summary

This notebook version is the direct replacement for the standalone script.

It keeps the whole rebuild on the bench:
- forward schedule
- reverse rebuild
- seam observables
- jumper diagnostic
- GF(2) geometry
- parity / branch scaffold
- seam augmentation
